In [1]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-49892f84100b4c73843a82e371fc3522"


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    "deepseek-flash",
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

hi, my name is bob
================================== Ai Message ==================================

Soft paws pad on silent floors,  
Whiskers twitch by closing doors.  
Sunbeam naps and midnight zoom,  
A tail that flicks around the room.  

Purrs like thunder, small and deep,  
Secrets kept in velvet sleep.  
Eyes of moon and emerald green—  
The quiet king, the graceful queen.
================================ Human Message =================================

now do the same but for dogs
================================== Ai Message ==================================

Loyal hearts and wagging tails,  
Greet the day with happy wails.  
Muddy paws and slobbery kisses,  
Fetching balls and chasing wishes.  

Barks like bells and joyful leaps,  
Guardian strong while family sleeps.  
Eyes of amber, brown, and blue—  
The faithful friend, the tried and true.
================================ Human Message ===